In [1]:
%pip install torch pandas numpy scikit-learn tqdm

In [5]:
!git clone https://github.com/imets01/synthetic_network_data_gen.git

Cloning into 'synthetic_network_data_gen'...
remote: Enumerating objects: 504, done.
remote: Counting objects: 100% (504/504), done.
remote: Compressing objects: 100% (247/247), done.
remote: Total 504 (delta 288), reused 456 (delta 251), pack-reused 0 (from 0)
Receiving objects: 100% (504/504), 3.89 MiB | 10.10 MiB/s, done.
Resolving deltas: 100% (288/288), done.


In [7]:
%cd synthetic_network_data_gen
!git fetch
!git checkout anna

/content/synthetic_network_data_gen
Branch 'anna' set up to track remote branch 'anna' from 'origin'.
Switched to a new branch 'anna'


In [2]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

In [13]:
class QuicSequenceDataset(Dataset):
    def __init__(self, high_level_csv, low_level_dir):
        self.high_level_df = pd.read_csv(high_level_csv)
        self.high_level_df = self.high_level_df.head(20)
        self.low_level_dir = low_level_dir

        not_to_use_for_training = ['initial_ip_client', 'initial_ip_server', 'initial_port_client', 'initial_port_server','time_first', 'time_last']
        self.condition_features = self.high_level_df.drop(not_to_use_for_training, axis=1)
        self.flow_ids = self.high_level_df['file_id']

        self.condition_scaler = MinMaxScaler()
        self.sequence_scaler = MinMaxScaler()

        self.scaled_conditions = self.condition_scaler.fit_transform(self.condition_features)

        all_sequences = []
        for flow_id in self.flow_ids:
            file_path = os.path.join(self.low_level_dir, f"quiche_capture_{flow_id}.csv")
            df = pd.read_csv(file_path)
            all_sequences.append(df.values)

        full_sequence_data = np.concatenate(all_sequences, axis=0)
        self.sequence_scaler.fit(full_sequence_data)

    def __len__(self):
        return len(self.high_level_df)

    def __getitem__(self, idx):
        flow_id = self.flow_ids[idx]
        file_path = os.path.join(self.low_level_dir, f"flow_{flow_id}.csv")

        sequence_df = pd.read_csv(file_path)
        scaled_sequence = self.sequence_scaler.transform(sequence_df.values)

        scaled_condition = self.scaled_conditions[idx]

        return {
            'sequence': torch.FloatTensor(scaled_sequence),
            'condition': torch.FloatTensor(scaled_condition)
        }

In [4]:
def collate_fn(batch):
    sequences = [item['sequence'] for item in batch]
    conditions = torch.stack([item['condition'] for item in batch])
    padded_sequences = nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=0.0)
    return {'sequence': padded_sequences, 'condition': conditions}

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim, condition_dim, sequence_feature_dim, hidden_dim=64):
        super().__init__()
        self.fc_hidden = nn.Linear(latent_dim + condition_dim, hidden_dim)
        self.rnn = nn.GRU(input_size=sequence_feature_dim, hidden_size=hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, sequence_feature_dim)

    def forward(self, noise, condition, seq_len):
        combined_input = torch.cat([noise, condition], dim=1)
        h0 = self.fc_hidden(combined_input).unsqueeze(0)

        batch_size = noise.shape[0]
        dummy_input = torch.zeros(batch_size, seq_len, self.rnn.input_size, device=noise.device)

        rnn_out, _ = self.rnn(dummy_input, h0)
        output = torch.tanh(self.fc_out(rnn_out))
        return output

class Discriminator(nn.Module):
    def __init__(self, condition_dim, sequence_feature_dim, hidden_dim=64):
        super().__init__()
        self.rnn = nn.GRU(input_size=sequence_feature_dim + condition_dim, hidden_size=hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, 1)

    def forward(self, sequence, condition):
        seq_len = sequence.shape[1]
        condition_expanded = condition.unsqueeze(1).repeat(1, seq_len, 1)
        combined_input = torch.cat([sequence, condition_expanded], dim=2)
        _, hn = self.rnn(combined_input)
        out = self.fc_out(hn.squeeze(0))
        return out


In [ ]:
def train_rgan(dataloader, generator, discriminator, g_optimizer, d_optimizer, loss_fn, device, epochs=100, latent_dim=10):
    for epoch in range(epochs):
        for i, batch in enumerate(tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")):
            real_sequences = batch['sequence'].to(device)
            conditions = batch['condition'].to(device)
            batch_size, seq_len, _ = real_sequences.shape

            # Train Discriminator
            d_optimizer.zero_grad()
            real_output = discriminator(real_sequences, conditions)
            loss_real = loss_fn(real_output, torch.ones_like(real_output))

            noise = torch.randn(batch_size, latent_dim, device=device)
            fake_sequences = generator(noise, conditions, seq_len)
            fake_output = discriminator(fake_sequences.detach(), conditions)
            loss_fake = loss_fn(fake_output, torch.zeros_like(fake_output))

            d_loss = (loss_real + loss_fake) / 2
            d_loss.backward()
            d_optimizer.step()

            # Train Generator
            g_optimizer.zero_grad()
            gen_output = discriminator(fake_sequences, conditions)
            g_loss = loss_fn(gen_output, torch.ones_like(gen_output))
            g_loss.backward()
            g_optimizer.step()

        print(f"Epoch [{epoch+1}/{epochs}], D Loss: {d_loss.item():.4f}, G Loss: {g_loss.item():.4f}")

In [ ]:
DATA_DIR = "quic_data"
BATCH_SIZE = 32
LATENT_DIM = 20
HIDDEN_DIM = 128
EPOCHS = 50
LR = 0.0002

In [9]:
high_level_csv = '/content/synthetic_network_data_gen/high_level_features/quiche/quiche_all_extracted_high_level_features.csv'
low_level_dir = '/content/synthetic_network_data_gen/low_level_features/quiche'

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

dataset = QuicSequenceDataset(high_level_csv, low_level_dir)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)


Using device: cuda


ValueError: could not convert string to float: 'IP_AND_PORT'

In [ ]:
# Get dimensions
condition_dim = dataset.scaled_conditions.shape[1]
sequence_feature_dim = dataset.sequence_scaler.n_features_in_

# 3. Instantiate models
generator = Generator(LATENT_DIM, condition_dim, sequence_feature_dim, HIDDEN_DIM).to(device)
discriminator = Discriminator(condition_dim, sequence_feature_dim, HIDDEN_DIM).to(device)

# 4. Optimizers and Loss
g_optimizer = torch.optim.Adam(generator.parameters(), lr=LR, betas=(0.5, 0.999))
d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=LR, betas=(0.5, 0.999))
loss_fn = nn.BCEWithLogitsLoss()

# 5. Train
train_rgan(dataloader, generator, discriminator, g_optimizer, d_optimizer, loss_fn, device, epochs=EPOCHS, latent_dim=LATENT_DIM)

# --- Step 5: How to Generate New Data ---
print("\n--- Generating a new sample ---")

generator.eval()
with torch.no_grad():
    sample_condition_scaled = dataset[0]['condition'].unsqueeze(0).to(device)
    noise = torch.randn(1, LATENT_DIM, device=device)
    desired_seq_len = 35

    generated_sequence_scaled = generator(noise, sample_condition_scaled, desired_seq_len)

    generated_sequence_scaled_cpu = generated_sequence_scaled.squeeze(0).cpu().numpy()
    generated_sequence_original_scale = dataset.sequence_scaler.inverse_transform(generated_sequence_scaled_cpu)

    print("Generated sequence shape:", generated_sequence_original_scale.shape)
    print("Sample of generated data (original scale):\n", pd.DataFrame(generated_sequence_original_scale, columns=['delta_time', 'packet_length', 'packet_direction']).head())